In [3]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("Loading raw NAV data...")
# Load the raw data
df_nav = pd.read_csv('../data/raw/02_nav_history.csv')

# 1. Parse dates to datetime
df_nav['date'] = pd.to_datetime(df_nav['date'])

# 2. Remove duplicates BEFORE setting the index
initial_rows = len(df_nav)
df_nav = df_nav.drop_duplicates(subset=['amfi_code', 'date'])
print(f"Dropped {initial_rows - len(df_nav)} duplicate rows.")

# 3. Validate NAV > 0
df_nav = df_nav[df_nav['nav'] > 0]

print("Resampling to fill holidays/weekends. This will be incredibly fast...")

# 4 & 5. Forward-fill missing NAV using native Pandas time-series resampling!
# First, set the date as the index so the resampler can read the calendar
df_nav = df_nav.set_index('date')

# Group by the fund, resample to a Daily ('D') frequency, and forward-fill
df_clean_nav = df_nav.groupby('amfi_code')['nav'].resample('D').ffill().reset_index()

# Clean up the sorting just to be safe
df_clean_nav = df_clean_nav.sort_values(by=['amfi_code', 'date']).reset_index(drop=True)

# Save to the new 'processed' folder!
output_path = '../data/processed/clean_nav.csv'
df_clean_nav.to_csv(output_path, index=False)

print(f"Task 1 Complete! Clean dataset saved to {output_path}")
print(f"Final shape: {df_clean_nav.shape[0]} rows, {df_clean_nav.shape[1]} columns")
df_clean_nav.head()

Loading raw NAV data...
Dropped 0 duplicate rows.
Resampling to fill holidays/weekends. This will be incredibly fast...
Task 1 Complete! Clean dataset saved to ../data/processed/clean_nav.csv
Final shape: 64320 rows, 3 columns


,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639


In [6]:
import pandas as pd
import re

print("Loading raw investor transactions...")
# Load the raw data 
df_trans = pd.read_csv('../data/raw/08_investor_transactions.csv')

# 1. Fix date formats (FIXED: using 'transaction_date')
print("Standardizing dates...")
df_trans['transaction_date'] = pd.to_datetime(df_trans['transaction_date'])

# 2. Standardise transaction_type using Regex/String matching
print("Cleaning transaction types...")
def clean_txn_type(txn):
    txn = str(txn).lower().strip()
    if re.search(r'sip', txn):
        return 'SIP'
    elif re.search(r'lump|one.?time', txn):
        return 'Lumpsum'
    elif re.search(r'redempt|withdraw|sell', txn):
        return 'Redemption'
    else:
        return 'Other'

df_trans['transaction_type'] = df_trans['transaction_type'].apply(clean_txn_type)

# 3. Validate amount > 0 (FIXED: using 'amount_inr')
initial_rows = len(df_trans)
df_trans = df_trans[df_trans['amount_inr'] > 0]
dropped_amounts = initial_rows - len(df_trans)
print(f"Dropped {dropped_amounts} invalid transactions (amount_inr <= 0).")

# 4. Check and standardise KYC status values
print("Standardizing KYC statuses...")
if 'kyc_status' in df_trans.columns:
    # Converts strings like 'verified ', 'PENDING' to clean Title Case
    df_trans['kyc_status'] = df_trans['kyc_status'].astype(str).str.strip().str.title()

# Save the cleaned dataset
output_path = '../data/processed/clean_transactions.csv'
df_trans.to_csv(output_path, index=False)

print(f"\nTask 2 Complete! Cleaned transactions saved to {output_path}")
print(f"Final shape: {df_trans.shape[0]} rows, {df_trans.shape[1]} columns")

# Preview the cleaned data
df_trans.head()

Loading raw investor transactions...
Standardizing dates...
Cleaning transaction types...
Dropped 0 invalid transactions (amount_inr <= 0).
Standardizing KYC statuses...

Task 2 Complete! Cleaned transactions saved to ../data/processed/clean_transactions.csv
Final shape: 32778 rows, 13 columns


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [9]:
import pandas as pd
import numpy as np

print("Loading raw scheme performance data...")
# FIXED: Using the exact filename from your directory!
df_perf = pd.read_csv('../data/raw/07_scheme_performance.csv')

print("Columns in dataset:", df_perf.columns.tolist())

# 1. Validate return values are numeric
print("Validating return metrics...")
return_cols = [col for col in df_perf.columns if 'return' in col.lower()]
for col in return_cols:
    df_perf[col] = pd.to_numeric(df_perf[col], errors='coerce')

# 2. Flag negative Sharpe ratios (Dynamically finding the column)
sharpe_cols = [col for col in df_perf.columns if 'sharpe' in col.lower()]
if sharpe_cols:
    s_col = sharpe_cols[0] # Grab the first column that has 'sharpe' in the name
    df_perf[s_col] = pd.to_numeric(df_perf[s_col], errors='coerce')
    df_perf['flag_negative_sharpe'] = df_perf[s_col] < 0
    print(f"Flagged {df_perf['flag_negative_sharpe'].sum()} funds with negative Sharpe ratios.")
else:
    print("⚠️ WARNING: 'sharpe_ratio' column not found.")

# 3. Check expense_ratio range (0.1% to 2.5%) (Dynamically finding the column)
expense_cols = [col for col in df_perf.columns if 'expense' in col.lower()]
if expense_cols:
    e_col = expense_cols[0]
    df_perf[e_col] = pd.to_numeric(df_perf[e_col], errors='coerce')
    initial_rows = len(df_perf)
    df_perf = df_perf[(df_perf[e_col] >= 0.1) & (df_perf[e_col] <= 2.5)]
    dropped = initial_rows - len(df_perf)
    print(f"Dropped {dropped} rows with out-of-bounds expense ratios.")
else:
    print("⚠️ WARNING: 'expense_ratio' column not found.")

# Save the cleaned dataset
output_path = '../data/processed/clean_performance.csv'
df_perf.to_csv(output_path, index=False)

print(f"\nTask 3 Complete! Cleaned performance data saved to {output_path}")
print(f"Final shape: {df_perf.shape[0]} rows, {df_perf.shape[1]} columns")

# Preview the cleaned data
df_perf.head()

Loading raw scheme performance data...
Columns in dataset: ['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']
Validating return metrics...
Flagged 0 funds with negative Sharpe ratios.
Dropped 0 rows with out-of-bounds expense ratios.

Task 3 Complete! Cleaned performance data saved to ../data/processed/clean_performance.csv
Final shape: 40 rows, 20 columns


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade,flag_negative_sharpe
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate,False
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate,False
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High,False
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High,False
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low,False


In [10]:
import pandas as pd
import sqlite3
from sqlalchemy import create_engine

print("1. Connecting to SQLite database...")
# This will automatically create the 'bluestock_mf.db' file in your data/db folder
db_path = '../data/db/bluestock_mf.db'
engine = create_engine(f'sqlite:///{db_path}')

print("2. Executing schema.sql to build the tables...")
# We use the native sqlite3 library to safely run multiple SQL commands at once
with open('../sql/schema.sql', 'r') as file:
    schema_script = file.read()

with sqlite3.connect(db_path) as conn:
    conn.executescript(schema_script)
print("✅ Empty tables successfully created.")

print("3. Loading data into the database (this might take a minute)...")

# Load 1: Dimension Table (From our raw master file we validated on Day 1)
print("   -> Loading dim_fund...")
df_master = pd.read_csv('../data/raw/01_fund_master.csv')
df_master.to_sql('dim_fund', con=engine, if_exists='append', index=False)

# Load 2: Fact NAV
print("   -> Loading fact_nav...")
df_nav = pd.read_csv('../data/processed/clean_nav.csv')
df_nav.to_sql('fact_nav', con=engine, if_exists='append', index=False)

# Load 3: Fact Transactions
print("   -> Loading fact_transactions...")
df_trans = pd.read_csv('../data/processed/clean_transactions.csv')
df_trans.to_sql('fact_transactions', con=engine, if_exists='append', index=False)

# Load 4: Fact Performance
print("   -> Loading fact_performance...")
df_perf = pd.read_csv('../data/processed/clean_performance.csv')
# Keep only the columns that match our SQL schema
perf_cols = ['amfi_code', 'return_1y', 'return_3y', 'return_5y', 'sharpe_ratio', 'flag_negative_sharpe', 'expense_ratio']
# Find which of these columns actually exist in our cleaned file
existing_cols = [col for col in perf_cols if col in df_perf.columns]
df_perf[existing_cols].to_sql('fact_performance', con=engine, if_exists='append', index=False)

print("\n🎉 Task 5 Complete! All data successfully loaded into bluestock_mf.db")

1. Connecting to SQLite database...
2. Executing schema.sql to build the tables...
✅ Empty tables successfully created.
3. Loading data into the database (this might take a minute)...
   -> Loading dim_fund...
   -> Loading fact_nav...
   -> Loading fact_transactions...
   -> Loading fact_performance...

🎉 Task 5 Complete! All data successfully loaded into bluestock_mf.db


In [11]:
import sqlite3
import pandas as pd

print("Connecting to bluestock_mf.db to run analytics...\n")
conn = sqlite3.connect('../data/db/bluestock_mf.db')

# Read the SQL file we just created
with open('../sql/queries.sql', 'r') as file:
    sql_script = file.read()

# Split the file into individual queries using the semicolon
queries = [q.strip() for q in sql_script.split(';') if q.strip()]

# Loop through and print the results for each query
for i, query in enumerate(queries, 1):
    print(f"=== QUERY {i} RESULTS ===")
    try:
        # Pandas read_sql_query executes the SQL and formats it as a clean table!
        result_df = pd.read_sql_query(query, conn)
        print(result_df.head(), "\n")
    except Exception as e:
        print(f"Error running query: {e}\n")

conn.close()

Connecting to bluestock_mf.db to run analytics...

=== QUERY 1 RESULTS ===
                                    scheme_name return_1y
0     SBI Bluechip Fund - Regular Plan - Growth      None
1      SBI Bluechip Fund - Direct Plan - Growth      None
2    SBI Small Cap Fund - Regular Plan - Growth      None
3     SBI Small Cap Fund - Direct Plan - Growth      None
4  SBI Magnum Gilt Fund - Regular Plan - Growth      None 

=== QUERY 2 RESULTS ===
     month     avg_nav
0  2026-05  356.990317
1  2026-04  355.034911
2  2026-03  347.210656
3  2026-02  342.034213
4  2026-01  337.072998 

=== QUERY 3 RESULTS ===
   year  total_sip_inflow
0  2024       153233052.0
1  2025        64000439.0 

=== QUERY 4 RESULTS ===
            state  txn_count  total_amount
0          Punjab       2965   315780459.0
1      Tamil Nadu       2806   315177237.0
2  Madhya Pradesh       2931   308312493.0
3       Rajasthan       2577   298645822.0
4         Gujarat       2780   298358940.0 

=== QUERY 5 RESULTS ===